# Chapter 7 gallery — robust logistic regression

Reproduces `leukemia.R` (Ex 7.1) and `skin.R` (Ex 7.2) via the `by_logreg` / `wby_logreg` / `wml_logreg` family.

In [ ]:
import os, sys, pathlib


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## leukemia — weighted Bianco–Yohai logistic regression (Example 7.1)

`leukemia.R` fits `logregWBY` (weighted BY M-estimator) to the leukemia survival data and compares deviance residuals with the ML fit. (The `robust::glmRob` cubif comparator is out of scope and omitted.)

In [ ]:
leuk = rpm.datasets.leuk_dat()
print('columns:', list(leuk.columns))
Xl = leuk.iloc[:, :2].to_numpy(dtype=float)
yl = leuk.iloc[:, -1].to_numpy(dtype=float)
wby = rpm.wby_logreg(Xl, yl, intercept=True)
print(wby)
print('coefficients:', np.round(wby.coefficients, 4))
print('std deviation:', np.round(wby.standard_deviation, 4))

### Strict-tier cross-check vs direct R `logregWBY`

In [ ]:
ro.globalenv['Xl'] = Xl
ro.globalenv['yl'] = yl.reshape(-1, 1)
ro.r('rw <- logregWBY(Xl, yl, intercept=1)')
r_coef = np.asarray(ro.r('as.numeric(rw$coefficients)'), dtype=float)
print('coefficients bit-equal to R:', np.array_equal(wby.coefficients, r_coef))

## skin — robust logistic regression family (Example 7.2)

`skin.R` fits the weighted-M (`logregWBY`), plain BY (`logregBY`) and weighted-ML (`logregWML`) estimators to the vaso-constriction data. We reproduce all three; the ML and cubif comparators are out of scope.

In [ ]:
skin = rpm.datasets.skin()
print('columns:', list(skin.columns))
Xs = skin.iloc[:, :2].to_numpy(dtype=float)
ys = skin['vasoconst'].to_numpy(dtype=float)
wby = rpm.wby_logreg(Xs, ys, intercept=True)
by = rpm.by_logreg(Xs, ys, intercept=True)
wml = rpm.wml_logreg(Xs, ys, intercept=True)
for name, fit in (('WBY', wby), ('BY', by), ('WML', wml)):
    print(f'{name:>3}: coef = {np.round(fit.coefficients, 4)}')

In [ ]:
# Figure 7.5 analogue: sorted |deviance residuals| of the weighted-M fit
dev = np.sort(np.abs(wby.residual_deviances))
pp = (np.arange(1, len(dev)+1) - 0.5) / len(dev)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pp, dev, 'o-', ms=4)
ax.set_xlabel('quantiles'); ax.set_ylabel('|deviance residuals|')
ax.set_title('skin — weighted-M deviance residuals (Fig 7.5 analogue)')
fig.savefig(FIG_DIR / 'ch7_skin.png', dpi=110, bbox_inches='tight'); plt.close(fig)
print('done')

## epilepsy — robust Poisson GLM (Example 7.3, Breslow data)

`epilepsy.R` fits the seizure-count model with several robust estimators. We reproduce **RQL/Mqle** and **MT** (`robustbase::glmrob`) and **CUBIF** (`robcbi::cubinf`), plus the ML `glm` baseline. The MATLAB-only **MP** estimator (no R code) is shown as the book's documented constants. Comparators only (`quantreg`, `robust::glmRob`) are omitted.

In [ ]:
ro.r("suppressMessages(library(robustbase)); data(breslow.dat, package='robust')")
ro.r("yy<-breslow.dat[,10]; xx1<-breslow.dat[,11]; xx2<-breslow.dat[,12]; xx3<-breslow.dat[,8]=='progabide'; xx4<-xx2*xx3; XX<-cbind(rep(1,59),xx1,xx2,xx3,xx4); colnames(XX)<-c('intercept','Age10','Base4','Progabide','interac.Base4-Progabide')")
epi = pd.DataFrame({
    "yy": np.asarray(ro.r("as.numeric(yy)"), float),
    "xx1": np.asarray(ro.r("as.numeric(xx1)"), float),
    "xx2": np.asarray(ro.r("as.numeric(xx2)"), float),
    "xx3": np.asarray(ro.r("as.logical(xx3)"), bool),
    "xx4": np.asarray(ro.r("as.numeric(xx4)"), float),
})
FORM = "yy ~ xx1 + xx2 + xx3 + xx4"
print(epi.head())

In [ ]:
# RQL / Mqle (deterministic) and MT (stochastic -> seed)
rql = rpm.glmrob(FORM, epi, family="poisson")
set_seed(11)
mt = rpm.glmrob(FORM, epi, family="poisson", method="MT")
# CUBIF on the explicit-intercept design
XX = np.asarray(ro.r("XX"), float)
yy = np.asarray(ro.r("as.numeric(yy)"), float)
Xdf = pd.DataFrame(XX, columns=[str(c) for c in ro.r("colnames(XX)")])
cub = rpm.cubinf(Xdf, yy, family="poisson", intercept=False, null_dev=False, ufact=1.1)
# ML baseline via R glm (comparator; no new Python wrapper)
ml_coef = np.asarray(ro.r("as.numeric(glm(yy~xx1+xx2+xx3+xx4, family=poisson)$coefficients)"), float)
# MATLAB MP estimator: documented constants (no R/Python code exists)
mp_coef = np.array([2.0078, 0.0707, 0.1346, -0.4898, 0.0476])

table = pd.DataFrame({
    "ML": ml_coef, "CUBIF": cub.coefficients, "MT": mt.coefficients,
    "RQL": rql.coefficients, "MP(MATLAB)": mp_coef,
}, index=["intercept","Age10","Base4","Progabide","Base4:Prog"])
print("Table 7.3 — coefficient estimates:"); print(table.round(4))

In [ ]:
# strict-tier checks vs direct R
ro.r("rRQL <- glmrob(yy~xx1+xx2+xx3+xx4, family=poisson)")
assert np.array_equal(rql.coefficients, np.asarray(ro.r("as.numeric(rRQL$coefficients)"), float))
ro.r("set.seed(11L); rMT <- glmrob(yy~xx1+xx2+xx3+xx4, family=poisson, method='MT')")
assert np.array_equal(mt.coefficients, np.asarray(ro.r("as.numeric(rMT$coefficients)"), float))
ro.r("rCUB <- robcbi::cubinf(XX, yy, family=poisson(), null.dev=FALSE, control=robcbi::cubinf.control(ufact=1.1))")
assert np.array_equal(cub.coefficients, np.asarray(ro.r("as.numeric(rCUB$coefficients)"), float))
print("strict-tier vs R (RQL, MT, CUBIF): OK")

In [ ]:
# Figure 7.6 analogue: boxplots of absolute deviance residuals
def dev_resid(y, fitted):
    return np.sign(y - fitted) * np.sqrt(2*(y*np.log(np.maximum(y,1)) - y - y*np.log(fitted) + fitted))
ml_fitted = np.asarray(ro.r("as.numeric(glm(yy~xx1+xx2+xx3+xx4, family=poisson)$fitted)"), float)
mp_fitted = np.exp(XX @ mp_coef)
devs = {"ML": dev_resid(yy, ml_fitted), "MT": dev_resid(yy, mt.fitted_values),
        "QL": dev_resid(yy, rql.fitted_values), "MP": dev_resid(yy, mp_fitted)}
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([np.abs(v) for v in devs.values()], labels=list(devs.keys()))
ax.set_ylabel("Absolute deviance residuals"); ax.set_title("epilepsy — robust GLM deviances (Fig 7.6)")
fig.savefig(FIG_DIR / "ch7_epilepsy_dev.png", dpi=110, bbox_inches="tight"); plt.close(fig)
print("Figure 7.6 saved — epilepsy.R reproduced.")